### 07. 문서 군집화 소개와 실습 (Opinion Review 데이터 세트)

#### 문서 군집화 개념 
문서 군집화는 비슷한 텍스트 구성의 문서를 군집화 하는 것. 

문서  군집화 : 동일한 군집에 속하는 문서를 같은 카테고리 소속으로 분류할 수 있기 때문에 텍스트 분류 기반의 문서 분류와 유사하다.
- 하지만 문서 군집화는 학습 데이터 세트가 없는 비지도 학습 기반으로 동작한다. 

In [19]:
import pandas as pd 
import glob, os
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 700)

path = '/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1.0/topics'

# path로 저장한 디렉터리 밑에 있는 모든 .data 파일들의 파일명을 리스트로 취합
all_files = glob.glob(os.path.join(path, "*.data"))
filename_list =[]
opinion_text = []

# 개별 파일들의 파일명은 filename_list 리스트로 취합
# 개별 파일들의 파일 내용은 dataframe 로딩 후 다시 string으로 변환하여 opinion_test 리스트로 취합 
for file_ in all_files :
    # 개별 파일을 읽어서 dataframe으로 생성
    df = pd.read_table(file_, index_col=None, header=0, encoding='latin1')
    # 절대경로로 주어진 파일명을 가공
    # 맨 마지막.data 확장지도 제거
    filename_=file_.split('\\')[-1]
    filename =filename_.split('.')[0]
    # 파일명 리스트와 파일 내용 리스트에 파일명과 파일 내용을 추가
    filename_list.append(filename)
    opinion_text.append(df.to_string())


document_df = pd.DataFrame({'filename':filename_list,'opinion_text':opinion_text})
document_df.head()

,filename,opinion_text
0,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,short battery life I moved up from an 8gb .\n0 I love this ipod except for the battery life .\n1 ...
1,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 ...
2,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"We arrived at 23,30 hours and they could not recommend a restaurant so we decided to go to Tesco, with very limited choices but when you are hingry you do not careNext day they rang the bell at 8,00 hours to clean the room, not being very nice being waken up so earlyEvery day they gave u..."
3,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"Great location for tube and we crammed in a fair amount of sightseeing in a short time .\n0 All in all, a normal chain hotel on a nice lo..."
4,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,Staff are friendl...


In [13]:
import nltk

from nltk.tokenize import word_tokenize

from nltk.stem import WordNetLemmatizer

nltk.download('punkt')

nltk.download('punkt_tab')

nltk.download('wordnet')
              
def preprocess_text(text):
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(token) for token in word_tokenize(text)]

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/seomichelle/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/seomichelle/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/seomichelle/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [15]:
import nltk
import string

from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize

nltk.download('punkt')
nltk.download('wordnet')

remove_punct_dict = dict(
    (ord(punct), None) for punct in string.punctuation
)

lemmar = WordNetLemmatizer()

def LemTokens(tokens):
    return [lemmar.lemmatize(token) for token in tokens]

def LemNormalize(text):
    return LemTokens(
        nltk.word_tokenize(
            text.lower().translate(remove_punct_dict)
        )
    )

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/seomichelle/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/seomichelle/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vect = TfidfVectorizer(tokenizer=LemNormalize, stop_words='english',  
                             ngram_range=(1,2), min_df=0.05, max_df=0.85)
# opinion_text 칼럼값으로 feature vectorization 수행 
feature_vect = tfidf_vect.fit_transform(document_df['opinion_text'])

In [21]:
from sklearn.cluster import KMeans 

# 5개 집합으로 군집화 수행. 예제를 위해 동일한 클러스터링 결과 도출용 random_state=0
km_cluster = KMeans(n_clusters=5, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)
cluster_label=km_cluster.labels_
cluster_centers = km_cluster.cluster_centers_


In [22]:
document_df['cluster_label'] = cluster_label
document_df.head()

,filename,opinion_text,cluster_label
0,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,short battery life I moved up from an 8gb .\n0 I love this ipod except for the battery life .\n1 ...,1
1,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 ...,4
2,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"We arrived at 23,30 hours and they could not recommend a restaurant so we decided to go to Tesco, with very limited choices but when you are hingry you do not careNext day they rang the bell at 8,00 hours to clean the room, not being very nice being waken up so earlyEvery day they gave u...",3
3,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"Great location for tube and we crammed in a fair amount of sightseeing in a short time .\n0 All in all, a normal chain hotel on a nice lo...",0
4,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,Staff are friendl...,3


In [23]:
document_df[document_df['cluster_label']==0].sort_values(by='filename')

,filename,opinion_text,cluster_label
3,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"Great location for tube and we crammed in a fair amount of sightseeing in a short time .\n0 All in all, a normal chain hotel on a nice lo...",0
13,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,Mediocre room and service for a very extravagant price .\n0 ...,0
16,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"Both of us having worked in tourism for over 14 years were very disappointed at the level of service provided by this gentleman .\n0 The service was good, very friendly staff and we loved the free wine reception each night .\n1 ...",0
17,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,The room was packed to capacity with queues at the food buffets .\n0 The over zealous staff cleared our unfinished drinks while we were collecting cooked food and movement around the room with plates was difficult in the crowded circumstances .\n1 ...,0
27,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"not customer, oriented hotelvery low service levelboor reception\n0 The room was quiet, clean, the bed and pillows were comfortable, and the serv...",0
28,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"All in all, a normal chain hotel on a nice location , I will be back if I do not find anthing closer to Picadilly for a better price .\n0 ...",0
32,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,The food for our event was delicious .\n0 ...,0
41,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"If a case was included, as with the Kindle 1, that would have been reflected in a higher price .\n0 lower overall price, with nice leather cover .\n1 ...",0


In [24]:
document_df[document_df['cluster_label']==1].sort_values(by='filename')

,filename,opinion_text,cluster_label
0,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,short battery life I moved up from an 8gb .\n0 I love this ipod except for the battery life .\n1 ...,1
5,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,The voice prompts and maps are wonderful especially when driving after dark .\n0 I also thought the the voice prompts of the 750 where more pleasant sounding than the 255w's .\n1 ...,1
6,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,Another feature on the 255w is a display of the posted speed limit on the road which you are currently on right above your current displayed speed .\n0 I found myself not even looking at my car speedometer as I could easily see my current speed and the speed limit of my route at a glance .\n1 ...,1
7,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"A few other things I'd like to point out is that you must push the micro, sized right angle end of the ac adapter until it snaps in place or the battery may not charge .\n0 The full size right shift k...",1
8,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,It is easy to read and when touching the screen it works great !\n0 and zoom out buttons on the 255w to the same side of the screen which makes it a bit easier .\n1 ...,1
9,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"After I plugged it in to my USB hub on my computer to charge the battery the charging cord design is very clever !\n0 After you have paged tru a 500, page book one, page, at, a, time to get from Chapter 2 to Chapter 15, see how excited you are about a low battery and all the time it took to get there !\n1 ...",1
10,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"It's fast to acquire satellites .\n0 If you've ever had a Brand X GPS take you on some strange route that adds 20 minutes to your trip, has you turn the wrong way down a one way road, tell you to turn AFTER you've passed the street, frequently loses the satellite signal, or has old maps missing streets, you know how important this stuff is .\n1 ...",1
11,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"6GHz 533FSB cpu, glossy display, 3, Cell 23Wh Li, ion Battery , and a 1 .\n0 Not to mention that as of now...",1
12,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,", I think the new keyboard rivals the great hp mini keyboards .\n0 Since the battery life difference is minimum, the only reason to upgrade would be to get the better keyboard .\n1 The keyboard is now as good as t...",1
14,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"I bought the 8, gig Ipod Nano that has the built, in video camera .\n0 Itunes has an on, line store, where you may purchase and download music and videos which will install onto the ipod .\n1 ...",1


In [25]:
document_df[document_df['cluster_label']==2].sort_values(by='filename')

,filename,opinion_text,cluster_label
22,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"First of all, the interior has way too many cheap plastic parts like the cheap plastic center piece that houses the clock .\n0 3 blown struts at 30,000 miles, interior trim coming loose and rattling squeaking, stains on paint, and bug splats taking paint off, premature uneven brake wear, on 3rd windsh...",2
42,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,I previously owned a Toyota 4Runner which had incredible build quality and reliability .\n0 I bought the Camry because of Toyota reliability and qua...,2
45,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,I love the new body style and the interior is a simple pleasure except for the center dash .\n0 ...,2


In [26]:
document_df[document_df['cluster_label']==3].sort_values(by='filename')

,filename,opinion_text,cluster_label
2,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"We arrived at 23,30 hours and they could not recommend a restaurant so we decided to go to Tesco, with very limited choices but when you are hingry you do not careNext day they rang the bell at 8,00 hours to clean the room, not being very nice being waken up so earlyEvery day they gave u...",3
4,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,Staff are friendl...,3
20,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"The staff at Swissotel were not particularly nice .\n0 Each time I waited at the counter for staff for several minutes and then was waved to the desk upon my turn with no hello or anything, or apology for waiting in line .\n1 ...",3
30,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"The Swissotel is one of our favorite hotels in Chicago and the corner rooms have the most fantastic views in the city .\n0 The rooms look like they were just remodled and upgraded, there was an HD TV and a nice iHome docking station to put my iPod so I could set the alarm to wake up with my music instead of the radio .\n1 ...",3
31,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"The room was not overly big, but clean and very comfortable beds, a great shower and very clean bathrooms .\n0 The second room was smaller, with a very inconvenient bathroom layout, but at least it was quieter and we were able to sleep .\n1 ...",3
39,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"Good Value good location , ideal choice .\n0 Great Location , Nice Rooms , Helpless Concierge\n1 ...",3
46,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"Great Location , Nice Rooms , H...",3
49,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,The wine reception is a great idea as it is nice to meet other travellers and great having access to the free Internet access in our room .\n0 They also have a computer available with free internet which is a nice bonus but I didn't find that out till the day before we left but was still able to get on there to check our flight to Vegas the next day .\n1 ...,3
50,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,Parking was expensive but I think this is common for San Fran .\n0 there is a fee for parking but well worth it seeing no where to park if you do have a car .\n1 ...,3


In [27]:
document_df[document_df['cluster_label']==4].sort_values(by='filename')

,filename,opinion_text,cluster_label
1,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 ...,4
18,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"Drivers seat not comfortable, the car itself compared to other models of similar class .\n0 ...",4
23,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"After slowing down, transmission has to be kicked to speed up .\n0 ...",4
29,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"Front seats are very uncomfortable .\n0 No memory seats, no trip computer, can only display outside temp with trip odometer .\n1 ...",4
35,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"It's quiet, get good gas mileage and looks clean inside and out .\n0 The mileage is great, and I've had to get used to stopping less for gas .\n1 Thought gas ...",4
43,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 Seats are fine, in fact of all the smaller sedans this is the most comfortable I found for the price as I am 6', 2 and 250# .\n1 Great gas mileage and comfortable on long trips ...",4
47,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"Very happy with my 08 Accord, performance is quite adequate it has nice looks and is a great long, distance cruiser .\n0 6, 4, 3 eco engine has poor performance and gas mileage of 22 highway .\n1 Overall performance is good but comfort level is poor .\n2 ...",4


In [28]:
from sklearn.cluster import KMeans
# 3개의 집합으로 군집화 
km_cluster = KMeans(n_clusters=3, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)
cluster_label = km_cluster.labels_

# 소속 클러스터를 claster_label 칼럼으로 할당하고 cluster_label 값으로 정렬
document_df['cluster_label'] = cluster_label
document_df.sort_values(by='cluster_label')

,filename,opinion_text,cluster_label
50,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,Parking was expensive but I think this is common for San Fran .\n0 there is a fee for parking but well worth it seeing no where to park if you do have a car .\n1 ...,0
27,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"not customer, oriented hotelvery low service levelboor reception\n0 The room was quiet, clean, the bed and pillows were comfortable, and the serv...",0
28,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"All in all, a normal chain hotel on a nice location , I will be back if I do not find anthing closer to Picadilly for a better price .\n0 ...",0
30,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"The Swissotel is one of our favorite hotels in Chicago and the corner rooms have the most fantastic views in the city .\n0 The rooms look like they were just remodled and upgraded, there was an HD TV and a nice iHome docking station to put my iPod so I could set the alarm to wake up with my music instead of the radio .\n1 ...",0
20,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"The staff at Swissotel were not particularly nice .\n0 Each time I waited at the counter for staff for several minutes and then was waved to the desk upon my turn with no hello or anything, or apology for waiting in line .\n1 ...",0
31,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"The room was not overly big, but clean and very comfortable beds, a great shower and very clean bathrooms .\n0 The second room was smaller, with a very inconvenient bathroom layout, but at least it was quieter and we were able to sleep .\n1 ...",0
32,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,The food for our event was delicious .\n0 ...,0
17,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,The room was packed to capacity with queues at the food buffets .\n0 The over zealous staff cleared our unfinished drinks while we were collecting cooked food and movement around the room with plates was difficult in the crowded circumstances .\n1 ...,0
16,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,"Both of us having worked in tourism for over 14 years were very disappointed at the level of service provided by this gentleman .\n0 The service was good, very friendly staff and we loved the free wine reception each night .\n1 ...",0
13,/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1,Mediocre room and service for a very extravagant price .\n0 ...,0


#### 군집별 핵심 단어 추출하기 

각 군집에 속한 문서는 핵심 단어를 주축으로 군집화돼 있을것이다. 이때 각 군집을 구성하는 핵심 단어가 어떤 것인지 확인

In [30]:
cluster_centers = km_cluster.cluster_centers_
print('cluster_centers shape:' , cluster_centers.shape)
print(cluster_centers)

cluster_centers shape: (3, 4610)
[[0.         0.00099548 0.00174656 ... 0.         0.00183397 0.00144581]
 [0.0100545  0.         0.         ... 0.00706288 0.         0.        ]
 [0.         0.00092552 0.         ... 0.         0.         0.        ]]


각 행의 배열 값은 각 군집 내의 4611개 피처의 위치가 개별 중심과 얼마나 가까운가를 상대 값으로 나타낸 것. 

0에서 1의 값을 가질 수 있으며 1에 가까울수록 중심과 가까운 값을 의미한다. 

In [31]:
# 군집별 top n 핵심 단어, 그 단어의 중심 위치 상댓값, 대상 파일명을 변환함. 

def get_cluster_details(cluster_model, cluster_data, feature_names, clusters_num, top_n_features=10):
    cluster_details = {}
    # cluster_centers array의 값이 큰 순으로 정렬된 인덱스 값을 반환 
    # 군집 중심점별 할당된 word 피처들의 거리값이 큰 순으로 값을 구하기 위함
    centroid_features_ordered_ind = cluster_model.cluster_centers_.argsort()[:, ::-1]

    #개별 군집별로 반복하면서 핵심 단어, 그 단어의 중심 위치 상댓값, 대상 파일명 입력
    for cluster_num in range(clusters_num):
        cluster_details[cluster_num] = {}
        cluster_details[cluster_num]['cluster']=cluster_num

        #cluster_centers_.argsort()[:,::-1]로 구한 인덱스를 이용해 top n 피처 단어를 구함. 
        top_feature_indexes = centroid_features_ordered_ind[cluster_num, :top_n_features]
        top_features = [feature_names[ind] for ind in top_feature_indexes]

        # top_feature_indexes를 이용해 해당 피처 단어의 중심 위치 상댓값 구함 
        top_feature_values = cluster_model.cluster_centers_[cluster_num, top_feature_indexes].tolist()

        # cluster_details 딕셔너리 객체에 개별 군집별 핵심단어와 중심위치 상댓값, 해당 파일명 입력 
        cluster_details[cluster_num]['top_features'] = top_features 
        cluster_details[cluster_num]['top_features_value'] = top_feature_values
        filenames = cluster_data[cluster_data['cluster_label']== cluster_num]['filename']
        filenames = filenames.values.tolist()

        cluster_details[cluster_num]['filenames']=filenames

    return cluster_details

In [39]:
def print_cluster_details(cluster_details):
    for cluster_num, cluster_detail in cluster_details.items():
        print('##### Cluster {0}'.format(cluster_num))
        print('Top Features:', cluster_detail['top_features'])
        print('Reviews 파일명 : ', cluster_detail['filenames'][:7])
        print('========================================')

In [40]:
feature_names = tfidf_vect.get_feature_names_out()

cluster_details = get_cluster_details(cluster_model=km_cluster, cluster_data=document_df, feature_names=feature_names, clusters_num=3, top_n_features=10 )

print_cluster_details(cluster_details)

##### Cluster 0
Top Features: ['room', 'hotel', 'service', 'staff', 'food', 'location', 'bathroom', 'clean', 'price', 'parking']
Reviews 파일명 :  ['/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1', '/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1', '/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1', '/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1', '/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1', '/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1', '/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1']
##### Cluster 1
Top Features: ['screen', 'battery', 'keyboard', 'battery life', 'life', 'kindle', 'direction', 'video', 'size', 'voice']
Reviews 파일명 :  ['/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1', '/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1', '/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1', '/Users/seomichelle/26-2 ESAA:Python/dataset/OpinosisDataset1'